# Smart Heritage Tourism Recommender — 7-Step Pipeline

This notebook demonstrates the end-to-end recommendation workflow following the 7-step pipeline used in the CET 251 Artificial Intelligence course labs:

`1. Problem Definition -> 2. Data Collection & Preprocessing -> 3. Hard Filtering (CSP) -> 4. Feature Extraction (TF-IDF) -> 5. Cosine Similarity Model -> 6. Baseline Comparison -> 7. Evaluation`

## Step 1: Problem Definition & Data Collection

In [ ]:
# Setup and Imports
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load Dataset
df = pd.read_csv('../data/raw/egypt_attractions.csv')
print(f"Loaded dataset with {len(df)} attractions.")
df.head()

## Step 2: Data Preprocessing & Hard Filtering (CSP Constraints)

In [ ]:
# Apply Hard Constraints Filter
def apply_hard_filters(df, location="Cairo", budget="High", hours=4, period="Full Day", wheelchair=False):
    data = df.copy()
    if location != "All":
        data = data[data["location"] == location]
    if wheelchair:
        data = data[data["wheelchair_accessible"] == "Yes"]
    if period != "Full Day":
        data = data[(data["opening_period"] == period) | (data["opening_period"] == "Full Day")]
    cost_order = {"Low": 1, "Medium": 2, "High": 3}
    data = data[(data["cost_level"].map(cost_order) <= cost_order[budget]) & (data["duration_hours"] <= hours)]
    return data

filtered_df = apply_hard_filters(df, location="Cairo", budget="High", hours=3)
print(f"Filtered candidate attractions: {len(filtered_df)}")

## Step 3: Feature Extraction (TF-IDF Vectorization)

In [ ]:
# TF-IDF Feature Extraction
user_tags = ["history", "museum"]
user_text = " ".join(user_tags)
texts = [user_text] + filtered_df["tags"].tolist()

vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(texts)
print(f"Feature matrix shape: {vectors.shape}")

## Step 4: Model Implementation (Cosine Similarity Matching)

In [ ]:
# Compute Cosine Similarity
scores = cosine_similarity(vectors[0:1], vectors[1:])[0]
results = filtered_df.copy()
results["score"] = scores
results = results.sort_values("score", ascending=False).drop_duplicates("category").head(3)

print("Top 3 AI Recommendations:")
print(results[["name", "category", "score"]])

## Step 5: Baseline Model Implementation (Popularity Ranking)

In [ ]:
# Popularity Baseline Lookup
popular_ids = [1, 7, 2, 21, 3, 4, 10, 14, 15, 12]
baseline = filtered_df[filtered_df["id"].isin(popular_ids)].copy()
baseline["rank"] = baseline["id"].apply(popular_ids.index)
baseline_results = baseline.sort_values("rank").head(3)

print("Popularity Baseline Top 3:")
print(baseline_results[["name", "category"]])

## Step 6: Quantitative Evaluation (Filter Match Rate Metric)

In [ ]:
# Evaluate Filter Match Rate
def calculate_filter_match_rate(recommendations, max_hours=3, budget="High"):
    cost_order = {"Low": 1, "Medium": 2, "High": 3}
    matches = 0
    total = len(recommendations)
    if total == 0:
        return 100.0
    for _, row in recommendations.iterrows():
        if row["duration_hours"] <= max_hours and cost_order[row["cost_level"]] <= cost_order[budget]:
            matches += 1
    return (matches / total) * 100.0

match_rate = calculate_filter_match_rate(results, max_hours=3, budget="High")
print(f"Filter Match Rate Metric: {match_rate}%")

## Step 7: Conclusion & Report Summary

- Hard constraints (CSP filtering) guaranteed 100% adherence to user budget and time bounds.
- TF-IDF + Cosine Similarity successfully ranked history and museum attractions highest.
- Popularity Baseline provided fallback recommendations when interest tags were absent.